# Libraries Import and Base Path initialization

In [ ]:
import os, cv2, json, math, pickle, random
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from ultralytics import YOLO
from boxmot.trackers.ocsort.ocsort import OcSort
import torch

from mmengine.config import Config
from mmengine.registry import MODELS
from mmengine.runner import load_checkpoint
from mmaction.apis import init_recognizer

# # This function will now work correctly because we are running from the cloned directory
from mmaction.utils import register_all_modules
register_all_modules(init_default_scope=True) # We set the scope manually later

#Local Base Path
base_path = ""

# === Dataset Paths ===
data_path = os.path.join(base_path, "practice_videos")
video_path = os.path.join(data_path, "Bryan_LR_Complete.mp4")

annotation_path = os.path.join(data_path, "Bryan_LR_Complete.json")

labels_path = os.path.join(data_path, "move_labels.json") # move class labels
filtered_labels_path = os.path.join(data_path, "move_labels_filtered.json") # selected/remapped move labels
skeleton_dataset = os.path.join(data_path, "skeleton_dataset.pkl") # STGCN++ dataset
output_dir = os.path.join(data_path, "frames")
kp_dir = os.path.join(data_path)

# === Load YOLO detection and pose models ===
yolo  = YOLO("runs/detect/practice_m/weights/best.pt")
yolo_pose = YOLO("yolo11l-pose.pt")
yolo.to("mps")
yolo_pose.to("mps")

# YOLO and ByteTrack character tracking

In [ ]:
def make_tracker():
    return OcSort(
        half=True,
        device="mps",
        max_age=90,
        min_hits=2,
        iou_threshold=0.15,
        det_thresh=0.20,
    )

tracker = make_tracker()

def yolo_detect(image):
    result = yolo(image, conf=0.5, iou=0.35, classes=[0])[0]

    # the results of yolo.predict contains list of object per frame it detects, for example image will have 1 object in the list
    # while video will have as many object in it as the video frames
    # we will access the first object as it is an image
    # Play with conf(minimum conf to be detected) and 
    # iou (how much the boxes can overlap to be considered the same object)
    if result.boxes is None or len(result.boxes) == 0:
        return np.empty((0, 6), dtype=np.float32)

    xyxy = result.boxes.xyxy.cpu().numpy() # convert boxes x1, y1, x2, y2 of selected object to numpy, then to int
    conf = result.boxes.conf.cpu().numpy().reshape(-1, 1) # convert boxes confidence of selected object to numpy
    cls = result.boxes.cls.cpu().numpy().reshape(-1, 1) # convert boxes class of selected object to numpy

    detections = np.hstack((xyxy, conf, cls))
    # stack boxes, conf, cls horizontally (it only accepts tuple so we encapsulate it with double ()
    return detections

def ocsort_tracking(detections, image): 
    if detections.shape[0] == 0:
        return None
    
    tracks = tracker.update(detections, image)
    if tracks is None or len(tracks) == 0:
        return None
    
    ids = tracks[:, 4].astype(int).reshape(-1, 1)
    boxes = tracks[:, :4].astype(int)
    id_box_array = np.hstack((ids, boxes))
    return id_box_array

def pad_box(h, w, box, padding=0):
    # box: [id, x1, y1, x2, y2]
    _, x1, y1, x2, y2 = box
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding)
    y2 = min(h, y2 + padding)

    return np.array([_, x1, y1, x2, y2], dtype=int)
    
def crop_roi(frame, box):
    # box: [id, x1, y1, x2, y2]
    _, x1, y1, x2, y2 = box
    return frame[y1:y2, x1:x2]

    # Matrix (Memory) Coordinates: NumPy thinks in (row, column).
    # Because images are processed top-to-bottom, a row corresponds to 
    # the vertical y position, and a column corresponds to the horizontal x position.

def is_timer_missing(frame, edge_threshold=50):
    """
    Checks the Tekken timer UI using Edge Detection.
    Returns True if the sharp metallic borders of the numbers are missing.
    """
    y1, y2 = 26, 86
    x1, x2 = 600, 680
    timer_roi = frame[y1:y2, x1:x2]
    
    gray = cv2.cvtColor(timer_roi, cv2.COLOR_BGR2GRAY)
    
    # cv2.Canny highlights sharp transitions. 
    # The silver border of the font will light up brilliantly here.
    edges = cv2.Canny(gray, 100, 200)
    
    # Count how many 'edge' pixels exist in that small box
    edge_count = cv2.countNonZero(edges)
    
    # If the count drops below the threshold, the timer is gone.
    return edge_count < edge_threshold

# YOLO Pose Function

In [ ]:
def empty_keypoints():
    return np.full((17, 3), np.nan, dtype=np.float32)

def box_iou_xyxy(box, boxes):
    box = np.asarray(box, dtype=np.float32)
    boxes = np.asarray(boxes, dtype=np.float32)
    if boxes.size == 0:
        return np.array([], dtype=np.float32)

    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)

    box_area = max(0, box[2] - box[0]) * max(0, box[3] - box[1])
    boxes_area = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
    union = box_area + boxes_area - inter
    return np.divide(inter, union, out=np.zeros_like(inter, dtype=np.float32), where=union > 0)

def choose_pose_candidate(boxes_xyxy, box_scores, target_box=None):
    if boxes_xyxy.shape[0] == 0:
        return 0
    if target_box is None:
        return int(np.argmax(box_scores))

    target_box = np.asarray(target_box, dtype=np.float32)
    ious = box_iou_xyxy(target_box, boxes_xyxy)

    target_center = np.array([(target_box[0] + target_box[2]) / 2, (target_box[1] + target_box[3]) / 2], dtype=np.float32)
    candidate_centers = np.column_stack(((boxes_xyxy[:, 0] + boxes_xyxy[:, 2]) / 2, (boxes_xyxy[:, 1] + boxes_xyxy[:, 3]) / 2))
    distances = np.linalg.norm(candidate_centers - target_center, axis=1)
    target_diag = max(1.0, np.linalg.norm([target_box[2] - target_box[0], target_box[3] - target_box[1]]))
    center_score = 1.0 - np.clip(distances / target_diag, 0.0, 1.0)

    match_score = (2.0 * ious) + center_score + (0.25 * box_scores)
    return int(np.argmax(match_score))

def track_box_to_roi_box(track_box, crop_box):
    _, tx1, ty1, tx2, ty2 = track_box
    _, cx1, cy1, _, _ = crop_box
    return np.array([tx1 - cx1, ty1 - cy1, tx2 - cx1, ty2 - cy1], dtype=np.float32)

def run_yolopose(image, input_size=192, conf=0.25, iou=0.35, target_box=None):
    if image is None or image.size == 0:
        return empty_keypoints()

    roi_height, roi_width = image.shape[:2]
    if roi_height == 0 or roi_width == 0:
        return empty_keypoints()

    result = yolo_pose(image, imgsz=input_size, conf=conf, iou=iou, verbose=False)[0]
    if result.keypoints is None or len(result.keypoints) == 0:
        return empty_keypoints()

    xy = result.keypoints.xy.cpu().numpy()
    kp_conf = result.keypoints.conf
    if kp_conf is None:
        scores = np.ones(xy.shape[:2], dtype=np.float32)
    else:
        scores = kp_conf.cpu().numpy()

    if xy.shape[0] == 0:
        return empty_keypoints()

    if result.boxes is not None and len(result.boxes) > 0:
        boxes_xyxy = result.boxes.xyxy.cpu().numpy()
        box_scores = result.boxes.conf.cpu().numpy()
        best_idx = choose_pose_candidate(boxes_xyxy, box_scores, target_box)
        best_idx = min(best_idx, xy.shape[0] - 1)
    else:
        mean_scores = np.nanmean(scores, axis=1)
        best_idx = 0 if np.all(np.isnan(mean_scores)) else int(np.nanargmax(mean_scores))

    keypoints = empty_keypoints()
    points_xy = xy[best_idx]
    point_scores = scores[best_idx]
    num_points = min(17, points_xy.shape[0])

    keypoints[:num_points, 0] = points_xy[:num_points, 1] / roi_height
    keypoints[:num_points, 1] = points_xy[:num_points, 0] / roi_width
    keypoints[:num_points, 2] = point_scores[:num_points]
    return keypoints

def denormalize_points(points, original_height, original_width, input_size=None):
    """
    Converts normalized YOLO-pose ROI keypoints back to ROI pixel coordinates.
    """
    y, x, c = points
    if np.isnan(y) or np.isnan(x):
        return (np.nan, np.nan, c)

    y_abs = y * original_height
    x_abs = x * original_width
    return (y_abs, x_abs, c)

def normalize_points_to_full_frame(kp_array, box, full_height, full_width):
    """
    Normalize array of keypoints to full frame size
    """
    _, x1, y1, x2, y2 = box
    roi_height = y2-y1
    roi_width = x2-x1

    kp_full = []
    for kp in kp_array:
        y_roi, x_roi, c = denormalize_points(kp, roi_height, roi_width)
        if np.isnan(y_roi) or np.isnan(x_roi):
            kp_full.append([np.nan, np.nan, c])
            continue

        y_full = (y_roi + y1) / full_height
        x_full = (x_roi + x1) / full_width
        kp_full.append([y_full, x_full, c])

    return np.array(kp_full)
        

def draw_keypoints(frame, keypoints, color, full_height, full_width, score_threshold=0.0):
    if keypoints is None:
        return

    for kp in keypoints:
        y, x, c = kp
        if not np.isfinite(y) or not np.isfinite(x):
            continue
        if np.isfinite(c) and c < score_threshold:
            continue

        y_px = int(np.clip(y * full_height, 0, full_height - 1))
        x_px = int(np.clip(x * full_width, 0, full_width - 1))
        cv2.circle(frame, (x_px, y_px), 3, color, thickness=2, lineType=cv2.LINE_AA)

def interpolate_points(player_kp):
    player_kp = np.array(player_kp)
    for kp in range(player_kp.shape[1]):
        for coord in range(player_kp.shape[2]):
            data = player_kp[:, kp, coord]
            nans = np.isnan(data) # nans mask example: [true, false, true] based on the positions
            if np.any(~nans):
                data[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(~nans), data[~nans])
            player_kp[:, kp, coord] = data
            print(data)
    return player_kp

# YOLO Pose Extraction

In [ ]:
input_size = 384
cap = cv2.VideoCapture(video_path)
print(video_path, os.path.exists(video_path))

player1_kp = []
player2_kp = []
player1_track = []  # [id, x1, y1, x2, y2] or NaNs per frame
player2_track = []  # [id, x1, y1, x2, y2] or NaNs per frame
other_track = []    # [id, x1, y1, x2, y2] or NaNs per frame
player_set = False
player1_id, player2_id = None, None
player1_box, player2_box, other_box = None, None, None
player1_kp_full, player2_kp_full, other_kp_full = None, None, None
frame_count = 0

missing_timer_frames = 0
buffer_limit = 10  # 10 frames ignores brief juggles, but catches the 15-frame practice reset
tracking_active = True

cinematic_frames = 0
cinematic_buffer = 5 # Wait 5 frames to confirm a zoom

while cap.isOpened(): # read every single frame of the video
    ret, frame_bgr = cap.read()
    if not ret:
        print("End of video")
        break

    original_height, original_width = frame_bgr.shape[:2]

    # --- 1. CHECK THE TIMER ---
    if is_timer_missing(frame_bgr):
        missing_timer_frames += 1
    else:
        missing_timer_frames = 0
        tracking_active = True # Timer is clearly visible, tracking is safe

    # --- HANDLE THE RESET STATE ---
    if missing_timer_frames > buffer_limit:
        
        # Only print and wipe memory the FIRST time we cross the threshold
        if tracking_active:
            print(f"Scene transition detected at frame {frame_count}. Pausing tracking...")
            tracking_active = False 
            
            tracker = make_tracker()
            # last_p1_box = None
            # last_p2_box = None
            # If using an OC-SORT instance, destroy/re-init it here.
            player_set = False

        # We are in a reset state (black screen or waiting for fade-in).
        # Append empty frames to keep your temporal arrays perfectly aligned for the pipeline.
        player1_kp.append(np.full((17, 3), np.nan))
        player2_kp.append(np.full((17, 3), np.nan))
        player1_track.append(np.full(5, np.nan, dtype=np.float32))
        player2_track.append(np.full(5, np.nan, dtype=np.float32))
        other_track.append(np.full(5, np.nan, dtype=np.float32))
        
        frame_count += 1
        
        # Skip the rest of the loop entirely. Do not run YOLO.
        continue


    # --- 2. THE CINEMATIC CHECK ---
    # Run your raw YOLO detection ONCE per frame
    detections = yolo_detect(frame_bgr) 
    id_box_array = ocsort_tracking(detections, frame_bgr) # 2d array containing id_box from p1 and 2

    if id_box_array is None or id_box_array.size == 0:
        player1_kp.append(np.full((17, 3), np.nan))
        player2_kp.append(np.full((17, 3), np.nan))
        player1_track.append(np.full(5, np.nan, dtype=np.float32))
        player2_track.append(np.full(5, np.nan, dtype=np.float32))
        other_track.append(np.full(5, np.nan, dtype=np.float32))
        frame_count += 1
        continue

    if id_box_array is not None and id_box_array.shape[0] >= 2 and not player_set:
        centers_x = (id_box_array[:,1] + id_box_array[:,3]) / 2
        order = np.argsort(centers_x)           # left -> right
        player1_id = int(id_box_array[order[0], 0])
        player2_id = int(id_box_array[order[1], 0]) 

        player_set = True

    p1_exist = any(id_box_array[:, 0] == player1_id)
    p2_exist = any(id_box_array[:, 0] == player2_id)

    other_mask = ~np.isin(id_box_array[:, 0], [player1_id, player2_id])
    has_other = np.any(other_mask)
    player1_track_frame = np.full(5, np.nan, dtype=np.float32)
    player2_track_frame = np.full(5, np.nan, dtype=np.float32)
    other_track_frame = np.full(5, np.nan, dtype=np.float32)
    
    if p1_exist:
        player1_box = id_box_array[id_box_array[:, 0] == player1_id][0]
        player1_track_frame = player1_box.astype(np.float32)
        # print("p1 box", player1_box)
        player1_box_padded = pad_box(original_height, original_width, player1_box)
        player1_roi = crop_roi(frame_bgr, player1_box_padded)
        player1_target_box = track_box_to_roi_box(player1_box, player1_box_padded)
        player1_kp_raw = run_yolopose(player1_roi, input_size, target_box=player1_target_box)
        player1_kp_full = normalize_points_to_full_frame(player1_kp_raw, player1_box_padded, original_height, original_width)
        player1_kp.append(player1_kp_full)
    else:
        player1_box = None
        player1_box_padded = None
        player1_kp_full = None
        player1_kp.append(np.full((17, 3), np.nan))

    if p2_exist:
        player2_box = id_box_array[id_box_array[:, 0] == player2_id][0]
        player2_track_frame = player2_box.astype(np.float32)
        player2_box_padded = pad_box(original_height, original_width, player2_box)
        player2_roi = crop_roi(frame_bgr, player2_box_padded)
        player2_target_box = track_box_to_roi_box(player2_box, player2_box_padded)
        player2_kp_raw = run_yolopose(player2_roi, input_size, target_box=player2_target_box)
        player2_kp_full = normalize_points_to_full_frame(player2_kp_raw, player2_box_padded, original_height, original_width)
        player2_kp.append(player2_kp_full)
    else:
        player2_box = None
        player2_box_padded = None
        player2_kp_full = None
        player2_kp.append(np.full((17, 3), np.nan))

    if has_other:
        other_box = id_box_array[other_mask][0]
        other_track_frame = other_box.astype(np.float32)
        other_box_padded = pad_box(original_height, original_width, other_box)
        other_roi = crop_roi(frame_bgr, other_box_padded)
        other_target_box = track_box_to_roi_box(other_box, other_box_padded)
        other_kp_raw = run_yolopose(other_roi, input_size, target_box=other_target_box)
        other_kp_full = normalize_points_to_full_frame(other_kp_raw, other_box_padded, original_height, original_width)
    else:
        other_box = None
        other_box_padded = None
        other_kp_full = None


    # Visualization: draw player 1 (green) and player 2 (red)
    color_p1 = (0, 255, 0)  # green (B, G, R)
    color_p2 = (0, 0, 255)  # red
    
    # Boxes + IDs
    if player1_box is not None:
        obj_id, x1, y1, x2, y2 = player1_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color_p1, thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p1, thickness=1, lineType=cv2.LINE_AA)
    
    if player2_box is not None:
        obj_id, x1, y1, x2, y2 = player2_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color_p2, thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p2, thickness=1, lineType=cv2.LINE_AA)
    
    if other_box is not None:
        obj_id, x1, y1, x2, y2 = other_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (255, 0, 0), thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p2, thickness=1, lineType=cv2.LINE_AA)
    
    # Keypoints
    draw_keypoints(frame_bgr, player1_kp_full, color_p1, original_height, original_width)
    draw_keypoints(frame_bgr, player2_kp_full, color_p2, original_height, original_width)

    player1_track.append(player1_track_frame)
    player2_track.append(player2_track_frame)
    other_track.append(other_track_frame)

    cv2.imshow('Process feed', frame_bgr)
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

    frame_count += 1

cap.release()

player1_kp = np.asarray(player1_kp, dtype=np.float32)
player2_kp = np.asarray(player2_kp, dtype=np.float32)
player1_track = np.asarray(player1_track, dtype=np.float32)
player2_track = np.asarray(player2_track, dtype=np.float32)
other_track = np.asarray(other_track, dtype=np.float32)

np.save(os.path.join(kp_dir, "player1_kp"), player1_kp)
np.save(os.path.join(kp_dir, "player2_kp"), player2_kp)
np.save(os.path.join(kp_dir, "player1_track"), player1_track)
np.save(os.path.join(kp_dir, "player2_track"), player2_track)
np.save(os.path.join(kp_dir, "other_track"), other_track)

> **DEPRECATED: Old chat summary**  
> Ringkasan diskusi lama; jangan dimasukkan sebagai bagian pipeline eksekusi.

Sure! Here's a concise summary of everything we discussed, formatted in markdown for easy reference:

---

## 📚 Summary: MMAction2 Skeleton Dataset Format & Preparation Steps

Skeleton-based Action Recognition in MMAction2 doesn’t require splitting the original video, but it **does require splitting the keypoint data** into action-based segments.

---

### 🧬 Dataset Format Overview (`.pkl`)

```python
{
  "split": {
    "train": ["clip1", "clip2", ...],
    "val": ["clip7", "clip8", ...],
    ...
  },
  "annotations": [
    {
      "frame_dir": "clip1",
      "label": 0,
      "img_shape": (1080, 1920),
      "original_shape": (1080, 1920),
      "total_frames": 87,
      "keypoint": np.ndarray([M, T, V, C]),
      "keypoint_score": np.ndarray([M, T, V])
    },
    ...
  ]
}
```

- **`frame_dir`**: Unique name for each clip
- **`label`**: Action class (int)
- **`img_shape` & `original_shape`**: Optional frame resolution
- **`total_frames`**: Frames in the segment
- **`keypoint`**: Shape `[M x T x V x C]` (people, frames, joints, coords)
- **`keypoint_score`**: Confidence for each keypoint `[M x T x V]`

---

### ⚙️ Steps to Prepare from a Long Video

If you already extracted full video keypoints:

1. **Use Annotations**  
   Get frame ranges for each action from your annotation file.

2. **Slice Keypoint Arrays**  
   Extract each action clip from the full keypoint array using its frame indices.

3. **Assign Clip Identifiers**  
   Name each segment like `clip001`, `clip002`, etc.

4. **Group into Splits**  
   Organize clip names into `'train'`, `'val'`, etc. inside the `split` dictionary.

5. **Build Annotations List**  
   For each clip, create a dictionary with all required fields and add it to `annotations`.

6. **Save to Pickle**  
   Combine `split` and `annotations` into a Python dict and save as `.pkl`.

---

Want me to build a sample Python script to help automate these steps? Happy to dive in! 💻

# Train STGCN++ all moves

In [ ]:
%cd "/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS"

# Build dataset from every annotated move plus a small idle class.
# V13 note: use raw/no-global-interpolation player*_kp.npy from the v12 extraction cell.
# Important: positive move clips are expanded to fixed 30-frame windows
# so training looks closer to real rolling-window inference.
# V13 tests a recovery-biased offset set: less startup than -6,0,6, but not as aggressive as 0,3,6.
# Optional later experiment / v14: change to --positive-offsets=0,3,6 for stronger recovery bias.

!.venv/bin/python make_all_moves_idle_dataset.py \
  --positive-window-size 30 \
  --positive-window-mode center \
  --positive-offsets=-3,0,6 \
  --idle-count 30 \
  --idle-window-size 30 \
  --idle-stride 15 \
  --exclude-margin 15 \
  --min-mean-score 0.35

%cd "/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/mmaction2"

!MPLCONFIGDIR=/private/tmp/matplotlib \
PYTORCH_ENABLE_MPS_FALLBACK=1 \
CUDA_VISIBLE_DEVICES=-1 \
../.venv/bin/python tools/train.py \
  configs/skeleton/stgcnpp/tekken_stgcn_all_moves_idle.py \
  --work-dir work_dirs/tekken_stgcn_v13_all_moves_idle_no_interp_offsets_m3_0_6_rerun

# Run Inference

In [2]:
%cd "/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS"

!.venv/bin/python tekken_video_inference.py \
  --video practice_videos/Bryan_LR_Complete.mp4 \
  --output practice_videos/tekken_stgcn_demo_v13_no_interp_offsets_m3_0_6_0.75.mp4 \
  --config mmaction2/configs/skeleton/stgcnpp/tekken_stgcn_all_moves_idle.py \
  --labels practice_videos/move_labels_all_moves_idle.json \
  --checkpoint mmaction2/work_dirs/tekken_stgcn_v13_fix_sls_annotation/best_acc_top1_epoch_22.pth \
  --pose-model yolo11l-pose.pt \
  --pose-imgsz 384 \
  --hide-labels idle \
  --predict-every 3 \
  --action-conf 0.75 \
  --stable-predictions 4 \
  --kp-interpolation none \
  --predictions-json "Bukti projek/end_to_end_predictions_epoch22.json"

/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS
Loads checkpoint by local backend from path: mmaction2/work_dirs/tekken_stgcn_v13_fix_sls_annotation/best_acc_top1_epoch_22.pth
/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/.venv/lib/python3.10/site-packages/mmengine/runner/checkpoint.py:347: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  checkpoint = torch.load(filename, map_location=map_location)
WARNING  Max age > max observations, increasing size of max observations...     
INFO     OcSort: det_thresh=0.2, max_age=90, max_obs=50, min_hits=2,            
         iou_threshold=0.15, per_class=False, asso_func=iou, min_conf=0.1,      
         delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01,             
         Q_s_scaling=0.0001                                                     
Input: practice_videos/Bryan_LR_Complete.mp4
Output: practice_video

# Evaluate Inference Against Validation Annotations

This scores only the saved validation windows. The result is an internal end-to-end evaluation, not an unseen-recording generalization test.

In [8]:
%cd "/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS"

!.venv/bin/python evaluate_video_annotations.py \
  --predictions "Bukti projek/end_to_end_predictions_epoch22.json" \
  --split val \
  --event-pre-tolerance 0 \
  --event-post-tolerance 20 \
  --output-json "Bukti projek/end_to_end_val_evaluation_epoch22.json" \
  --confusion-csv "Bukti projek/end_to_end_val_confusion_epoch22.csv"

/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS
Evaluation split: val
Matched windows: 100/102 (unmatched 2)
Window metrics: Top-1=0.7500, Top-5=0.9900, macro-F1=0.7306, loss=0.7026, coverage=0.9804, strict Top-1 including missing=0.7353
Event metrics (mean probability across offsets): Top-1=0.7812, macro-F1=0.7688
Annotated-event detection (thresholded raw label): 26/32 (0.8125)
Annotated-event detection (stable displayed label): 19/32 (0.5938)
Saved evaluation: Bukti projek/end_to_end_val_evaluation_epoch22.json
Saved confusion matrix: Bukti projek/end_to_end_val_confusion_epoch22.csv


# Test inference

In [ ]:
%cd "/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS"

!.venv/bin/python tekken_video_inference.py \
  --video practice_videos/Bryan_LR_Complete.mp4 \
  --output "Bukti projek/e2e_benchmark_normal_power.mp4" \
  --metrics-json "Bukti projek/e2e_benchmark_normal_power.json" \
  --max-frames 330 \
  --metrics-warmup 30 \
  --metrics-window 60 \
  --device mps \
  --action-device cpu \
  --realtime-target-fps 0 \
  --device-label "MacBook Pro, Apple M5, 24 GB"

In [2]:
import cv2

video_path = "practice_videos/Bryan_LR_Complete.mp4"
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = frame_count / fps

print(f"FPS: {fps:.5f}")
print(f"Jumlah frame: {frame_count}")
print(f"Durasi: {duration:.2f} detik")

cap.release()

FPS: 29.97003
Jumlah frame: 7665
Durasi: 255.76 detik
